## **Installs**

In [ ]:
#!pip install -U transformers

## **Hugging Face Token**

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

In [ ]:
from huggingface_hub import login
login(token=hf_token)

## **Imports**

In [ ]:
import json
import re
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc, traceback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
MODEL_NAME = "google/gemma-2b"
HF_TOKEN = hf_token

## **Load Model**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

In [ ]:
#freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

## **Dataset Loading (with negatives)**

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/pico_dataset_with_negatives.csv")  # update path if running locally
# If running outside Kaggle, point this at your local copy instead, e.g.:
# df = pd.read_csv("pico_dataset_with_negatives.csv")
print(df.shape)
df.head(3)

## **Build (prompt, positive, negative) triples + train/val split**

Same minimal, un-instructed prompt as exp 1 (just the bare question). `positive`
is the PICO-formatted ground truth (P/I/C/O columns); `negative` is the
`negative_response` column you generated earlier.

In [ ]:
def format_positive(row):
    return (
        f"P: {row['P']}\n"
        f"I: {row['I']}\n"
        f"C: {row['C']}\n"
        f"O: {row['O']}"
    )

def format_prompt(row):
    return f"{row['research_question'].strip()}\n"

examples = []
for _, row in df.iterrows():
    examples.append({
        "prompt": format_prompt(row),
        "positive": format_positive(row) + tokenizer.eos_token,
        "negative": str(row["negative_response"]).strip() + tokenizer.eos_token,
    })

random.shuffle(examples)
n_val = max(5, int(0.1 * len(examples)))
val_examples = examples[:n_val]
train_examples = examples[n_val:]
print(f"train={len(train_examples)}  val={len(val_examples)}")
print("--- sample ---")
print("PROMPT:", train_examples[0]["prompt"])
print("POSITIVE:", train_examples[0]["positive"])
print("NEGATIVE:", train_examples[0]["negative"])

## **Steering vector module + forward hook**

Identical parameterization to exp 1: a single rank-1 direction `w1` + bias `b1`
hooked into one decoder layer's residual stream, `h <- h + alpha * w1 + b1`.
Everything else in the model stays frozen; only `w1`/`b1` get gradients.

In [ ]:
class SteeringVector(nn.Module):
    """Rank-1 steering vector: h <- h + alpha * w1 + b1  (eq. 9 in RePS paper)."""
    def __init__(self, dim, dtype=torch.float32):
        super().__init__()
        self.w1 = nn.Parameter(torch.randn(dim, dtype=dtype) * 0.01)
        self.b1 = nn.Parameter(torch.zeros(dim, dtype=dtype))

    def forward(self, h, alpha):
        # keep params in fp32 (see exp1 NaN fix) and cast to h's device/dtype
        # only at the point of injection into the model's (fp16) activations.
        w1 = self.w1.to(device=h.device, dtype=h.dtype)
        b1 = self.b1.to(device=h.device, dtype=h.dtype)
        return h + alpha * w1 + b1


class SteeringHook:
    """Forward hook that injects a SteeringVector into a decoder layer's output."""
    def __init__(self, steering_vector: SteeringVector):
        self.sv = steering_vector
        self.enabled = False
        self.alpha = 0.0
        self.handle = None

    def __call__(self, module, inputs, output):
        if not self.enabled:
            return output
        if isinstance(output, tuple):
            hidden_states = self.sv(output[0], self.alpha)
            return (hidden_states,) + output[1:]
        else:
            return self.sv(output, self.alpha)

    def attach(self, layer_module):
        self.handle = layer_module.register_forward_hook(self)
        return self.handle

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

In [ ]:
STEER_LAYER = max(1, int(NUM_LAYERS * 0.4))
layer_device = next(model.model.layers[STEER_LAYER].parameters()).device
print(f"Hooking layer index {STEER_LAYER} / {NUM_LAYERS} (on {layer_device})")

steering_vector = SteeringVector(HIDDEN_DIM, dtype=torch.float32).to(layer_device)
steering_hook = SteeringHook(steering_vector)
steering_hook.attach(model.model.layers[STEER_LAYER])

n_trainable = sum(p.numel() for p in steering_vector.parameters() if p.requires_grad)
print(f"Trainable steering-vector params: {n_trainable}  (2 x {HIDDEN_DIM})")

## **Preference objective: prefer the positive response over the negative one**

For a given `alpha`, we compute the (length-normalized) log-likelihood of the
*positive* response and of the *negative* response under the **same** steered
model, then push the vector to increase that margin:

`loss = -log sigmoid( BETA * ( logp(positive) - logp(negative) ) )`

This is a reference-free, single-direction version of the RePS/BiPO preference
loss (paper eq. 3-5) -- we only train the "steer toward positive" direction
(no separate null/suppression intervention), since that's what you asked for.

In [ ]:
def seq_logprob(prompt_ids, response_ids, alpha, device):
    """Length-normalized log p(response | prompt) under the steered model."""
    input_ids = torch.cat([prompt_ids, response_ids], dim=0).unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)

    steering_hook.enabled = True
    steering_hook.alpha = alpha
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits  # (1, seq_len, vocab)

    shift_logits = logits[:, :-1, :].float()
    shift_labels = input_ids[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_logp = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)  # (1, seq_len-1)

    P = prompt_ids.shape[0]
    R = response_ids.shape[0]
    resp_logp = token_logp[:, P - 1 : P - 1 + R]  # only the response tokens
    return resp_logp.sum() / R  # mean log-prob per token (SimPO-style length norm)


def build_pair(example, device):
    prompt_ids = tokenizer(example["prompt"], return_tensors="pt", add_special_tokens=True).input_ids[0]
    pos_ids = tokenizer(example["positive"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    neg_ids = tokenizer(example["negative"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    return prompt_ids, pos_ids, neg_ids

In [ ]:
TRAIN_FACTORS = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0]
LR = 0.04
EPOCHS = 12
GRAD_CLIP = 1.0
BETA = 1.0  # sharpness of the preference loss

optimizer = torch.optim.AdamW(steering_vector.parameters(), lr=LR, weight_decay=0.0)

def preference_loss(example, alpha, device):
    prompt_ids, pos_ids, neg_ids = build_pair(example, device)
    logp_pos = seq_logprob(prompt_ids, pos_ids, alpha, device)
    logp_neg = seq_logprob(prompt_ids, neg_ids, alpha, device)
    margin = logp_pos - logp_neg
    loss = -F.logsigmoid(BETA * margin)
    return loss, margin.item()

@torch.no_grad()
def evaluate(examples, alpha, device):
    margins, losses = [], []
    for ex in examples:
        prompt_ids, pos_ids, neg_ids = build_pair(ex, device)
        logp_pos = seq_logprob(prompt_ids, pos_ids, alpha, device)
        logp_neg = seq_logprob(prompt_ids, neg_ids, alpha, device)
        margin = (logp_pos - logp_neg).item()
        margins.append(margin)
        losses.append(-F.logsigmoid(torch.tensor(BETA * margin)).item())
    acc = float(np.mean([m > 0 for m in margins]))  # how often positive is preferred
    return float(np.mean(losses)), float(np.mean(margins)), acc

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_margin": [], "val_acc": []}
skipped_steps = 0

for epoch in range(EPOCHS):
    random.shuffle(train_examples)
    epoch_losses = []
    for ex in train_examples:
        alpha = random.choice(TRAIN_FACTORS)

        optimizer.zero_grad()
        loss, _ = preference_loss(ex, alpha, layer_device)

        if not torch.isfinite(loss):
            skipped_steps += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(steering_vector.parameters(), GRAD_CLIP)
        optimizer.step()
        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    val_loss, val_margin, val_acc = evaluate(val_examples, alpha=TRAIN_FACTORS[len(TRAIN_FACTORS)//2], device=layer_device)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_margin"].append(val_margin)
    history["val_acc"].append(val_acc)
    print(f"epoch {epoch+1:02d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_margin={val_margin:+.4f}  val_pref_acc={val_acc:.2f}  (skipped {skipped_steps})")

## **Select the best steering factor on held-out validation examples**

In [ ]:
candidate_factors = [1.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 16.0, 20.0]
factor_stats = {}
for f in candidate_factors:
    val_loss, val_margin, val_acc = evaluate(val_examples, alpha=f, device=layer_device)
    factor_stats[f] = {"loss": val_loss, "margin": val_margin, "acc": val_acc}
    print(f"alpha={f:>5.1f}  val_loss={val_loss:.4f}  val_margin={val_margin:+.4f}  val_pref_acc={val_acc:.2f}")

BEST_ALPHA = max(factor_stats, key=lambda f: factor_stats[f]["margin"])
print(f"\nBest steering factor: {BEST_ALPHA}")

## **Generation helper (steered vs. unsteered, same model / same weights)**

In [ ]:
@torch.no_grad()
def generate(prompt, alpha=None, max_new_tokens=100):
    steering_hook.enabled = alpha is not None
    steering_hook.alpha = alpha if alpha is not None else 0.0

    inputs = tokenizer(prompt, return_tensors="pt").to(layer_device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    steering_hook.enabled = False
    return text

In [ ]:
for ex in val_examples[:3]:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)
    print("UNSTEERED:")
    print(generate(ex["prompt"], alpha=None))
    print("-" * 80)
    print(f"STEERED (learned vector, alpha={BEST_ALPHA}):")
    print(generate(ex["prompt"], alpha=BEST_ALPHA))
    print("-" * 80)
    print("POSITIVE (ground truth):")
    print(ex["positive"].replace(tokenizer.eos_token, ""))

## **Save the learned steering vector**

In [ ]:
torch.save(
    {
        "w1": steering_vector.w1.detach().cpu(),
        "b1": steering_vector.b1.detach().cpu(),
        "layer": STEER_LAYER,
        "best_alpha": BEST_ALPHA,
        "model_name": MODEL_NAME,
        "objective": "preference (positive vs negative)",
    },
    "/kaggle/working/pico_steering_vector_exp2.pt",
)
print("Saved /kaggle/working/pico_steering_vector_exp2.pt")